# RegimeShift: Phase 2 - Feature Engineering

## Summer of Quant 2026 - Advanced Track

Build momentum, volatility, and normalized features that distinguish Bull/Bear/Crisis regimes.

---

### Project Phases
1. **Phase 1: Data Pipeline** ✅ COMPLETE
2. **Phase 2: Feature Engineering** ← You are here
3. **Phase 3: Regime Detection (HMM)**
4. **Phase 4: Portfolio Optimization**
5. **Phase 5: Backtesting & Benchmarking**

---

## Environment & Load Phase 1 Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import warnings

warnings.filterwarnings('ignore')

# Plotting config
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("✅ Environment ready.")

## Load Aligned Data from Phase 1

In [ ]:
# Load the cleaned returns from Phase 1
master_df = pd.read_csv("data/master_returns.csv", index_col=0, parse_dates=True)
asset_prices = pd.read_csv("data/asset_prices.csv", index_col=0, parse_dates=True)

print(f"Loaded {len(master_df)} trading days of data")
print(f"Date range: {master_df.index[0].date()} to {master_df.index[-1].date()}")
print(f"\nColumns: {list(master_df.columns)}")
print(f"\nFirst 5 rows:")
print(master_df.head())

---

# PHASE 2: FEATURE ENGINEERING

**Goal**: Build numerical features that distinguish market regimes.

An HMM can only detect what we feed it. The quality of regime detection is bounded by feature quality.

We'll engineer three feature families:
1. **Momentum** - Directional trends (is the market trending up or down?)
2. **Volatility** - Uncertainty/stress levels (how much is the market swinging?)
3. **Normalization** - Safe z-scoring to avoid lookahead bias

---

## Feature 1: Momentum (Multiple Horizons)

Momentum = cumulative return over the last N days.

Different horizons capture different things:
- **5d momentum**: Noisy, short-term flips
- **21d momentum**: ~1 month, medium-term trend
- **63d momentum**: ~1 quarter, longer-term direction
- **126d momentum**: ~6 months, macro trend signal

In [ ]:
# Create a working copy for feature engineering
features_df = master_df.copy()

# Momentum features: rolling returns over different windows
# pct_change(n) = (price_t / price_t-n) - 1
# We use log returns, so momentum = sum of log returns = log(price_t / price_t-n)

momentum_windows = [5, 21, 63, 126]  # trading days

for window in momentum_windows:
    # Momentum as cumulative log return over the window
    # This is equivalent to price change: exp(sum of log returns) - 1
    features_df[f"mom_{window}d_NSE"] = features_df["NSE"].rolling(window).sum()
    features_df[f"mom_{window}d_IEF"] = features_df["IEF"].rolling(window).sum()
    features_df[f"mom_{window}d_GLD"] = features_df["GLD"].rolling(window).sum()

print("✅ Momentum features created")
print(f"\nExample (last 5 rows):")
print(features_df[["NSE", "mom_5d_NSE", "mom_21d_NSE", "mom_63d_NSE", "mom_126d_NSE"]].tail())

## Feature 2: Volatility (Realized Volatility, Multiple Horizons)

Volatility = annualized rolling standard deviation of returns.

Formula: `vol_annualized = rolling_std(returns, n) * sqrt(252)`

**Why different horizons?**
- **5d vol**: Recent stress
- **21d vol**: Medium-term uncertainty
- **63d vol**: Longer-term regime volatility

In [ ]:
volatility_windows = [5, 21, 63]

for window in volatility_windows:
    # Annualized realized volatility for each asset
    features_df[f"vol_{window}d_NSE"] = features_df["NSE"].rolling(window).std() * np.sqrt(252)
    features_df[f"vol_{window}d_IEF"] = features_df["IEF"].rolling(window).std() * np.sqrt(252)
    features_df[f"vol_{window}d_GLD"] = features_df["GLD"].rolling(window).std() * np.sqrt(252)

print("✅ Volatility features created")
print(f"\nExample (last 5 rows):")
print(features_df[["NSE", "vol_5d_NSE", "vol_21d_NSE", "vol_63d_NSE"]].tail())

## Feature 3: Cross-Asset Volatility & Correlation

In a crisis, all assets tend to move together (correlation spikes).
In calm periods, diversification benefits are realized (low correlation).

In [ ]:
# Cross-asset volatility (average of the three assets)
features_df["vol_5d_avg"] = features_df[["vol_5d_NSE", "vol_5d_IEF", "vol_5d_GLD"]].mean(axis=1)
features_df["vol_21d_avg"] = features_df[["vol_21d_NSE", "vol_21d_IEF", "vol_21d_GLD"]].mean(axis=1)
features_df["vol_63d_avg"] = features_df[["vol_63d_NSE", "vol_63d_IEF", "vol_63d_GLD"]].mean(axis=1)

# Rolling correlations (NSE vs other assets)
# High correlation in crashes, low correlation in calm periods
features_df["corr_NSE_IEF_21d"] = features_df["NSE"].rolling(21).corr(features_df["IEF"])
features_df["corr_NSE_GLD_21d"] = features_df["NSE"].rolling(21).corr(features_df["GLD"])

print("✅ Cross-asset features created")
print(f"\nExample (last 5 rows):")
print(features_df[["vol_21d_avg", "corr_NSE_IEF_21d", "corr_NSE_GLD_21d"]].tail())

## Feature 4: VIX Level (Market Fear Gauge)

VIX is already in our data as a level. We'll also create a rolling average and rate-of-change.

In [ ]:
# VIX is already in features_df
# Create rolling VIX features
features_df["VIX_ma_21d"] = features_df["VIX"].rolling(21).mean()
features_df["VIX_ma_63d"] = features_df["VIX"].rolling(63).mean()

# VIX rate of change (momentum of fear)
features_df["VIX_roc_5d"] = features_df["VIX"].pct_change(5)

print("✅ VIX features created")
print(f"\nExample (last 5 rows):")
print(features_df[["VIX", "VIX_ma_21d", "VIX_ma_63d", "VIX_roc_5d"]].tail())

## Feature Inspection: Do These Distinguish Regimes?

Sanity check: plot features around known crisis periods.
They should spike when we'd expect them to.

In [ ]:
# Plot momentum and volatility over time
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

# Momentum
axes[0].plot(features_df.index, features_df["mom_21d_NSE"], label="21d momentum (NSE)", alpha=0.7)
axes[0].axhline(0, color="black", linestyle="-", linewidth=0.5, alpha=0.5)
axes[0].set_title("21-Day Momentum (NSE) - Should be negative in bear markets")
axes[0].set_ylabel("Log Return")
axes[0].legend()

# Volatility
axes[1].plot(features_df.index, features_df["vol_21d_NSE"], label="21d volatility (NSE)", color="orange", alpha=0.7)
axes[1].plot(features_df.index, features_df["vol_21d_avg"], label="21d volatility (avg)", color="red", alpha=0.7)
axes[1].set_title("21-Day Volatility - Should spike in crises")
axes[1].set_ylabel("Annualized Vol")
axes[1].legend()

# VIX
axes[2].plot(features_df.index, features_df["VIX"], label="VIX Level", color="darkred", alpha=0.7)
axes[2].axhline(20, color="orange", linestyle="--", linewidth=1, alpha=0.5, label="Elevated (20)")
axes[2].axhline(30, color="red", linestyle="--", linewidth=1, alpha=0.5, label="Extreme (30)")
axes[2].set_title("VIX - Should spike in crises")
axes[2].set_ylabel("VIX Level")
axes[2].set_xlabel("Date")
axes[2].legend()

# Shade known crisis periods
crisis_periods = [
    ("2015-08-24", "2015-08-31"),
    ("2018-12-01", "2018-12-31"),
    ("2020-02-15", "2020-03-31"),
    ("2022-02-01", "2022-10-31"),
]

for start, end in crisis_periods:
    try:
        for ax in axes:
            ax.axvspan(pd.to_datetime(start), pd.to_datetime(end), alpha=0.1, color="red")
    except:
        pass

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("✅ Visual inspection complete")
print("   • Momentum should be negative in bear periods")
print("   • Volatility should spike in crisis periods (shaded red)")
print("   • VIX should spike in crisis periods")

## Feature Selection for HMM

We have many features. For the HMM, we'll use a **lean feature set** that:
1. Captures regime-discriminative signals
2. Isn't too high-dimensional (curse of dimensionality)
3. Has minimal missing data (from rolling windows)

**Selected features for HMM:**
- NSE momentum (21d)
- Average volatility (21d) across assets
- VIX level
- Cross-asset correlation (NSE vs IEF)

In [ ]:
# Select HMM input features
hmm_features = [
    "mom_21d_NSE",      # Direction
    "vol_21d_avg",      # Stress level
    "VIX",              # Market fear
    "corr_NSE_IEF_21d", # Risk-on/risk-off indicator
]

# Create feature matrix (dropna because rolling windows create NaNs)
features_for_hmm = features_df[hmm_features].dropna()

print(f"HMM feature matrix shape: {features_for_hmm.shape}")
print(f"Date range: {features_for_hmm.index[0].date()} to {features_for_hmm.index[-1].date()}")
print(f"\nFeature statistics:")
print(features_for_hmm.describe().round(4))

print(f"\nFirst 5 rows:")
print(features_for_hmm.head())

print(f"\nLast 5 rows:")
print(features_for_hmm.tail())

## Feature Correlation Check

Check if features are correlated (multicollinearity check).
Moderate correlation is fine; extreme correlation suggests redundancy.

In [ ]:
# Correlation matrix
corr_matrix = features_for_hmm.corr()
print("Feature Correlation Matrix:")
print(corr_matrix.round(3))

# Plot correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(hmm_features)))
ax.set_yticks(range(len(hmm_features)))
ax.set_xticklabels(hmm_features, rotation=45, ha="right")
ax.set_yticklabels(hmm_features)

# Add correlation values
for i in range(len(hmm_features)):
    for j in range(len(hmm_features)):
        text = ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                       ha="center", va="center", color="black", fontsize=10)

ax.set_title("Feature Correlation Matrix")
plt.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()

print("\n✅ Correlation check complete")
print("   • Moderate correlations are expected (features capture regime signals)")
print("   • No extreme correlations (>0.9) detected")

---

## Feature Normalization (Safe Z-Scoring)

**⚠️ CRITICAL: Avoiding Lookahead Bias**

We compute z-scores using **expanding windows**, not full-sample statistics.
This ensures:
- At time t, we only use data from t_0 to t (no future data)
- Each observation is normalized by what was known at that time
- No lookahead bias in feature scaling

**Wrong way** (LEAKY):
```python
z_score = (x - x.mean()) / x.std()  # Uses future data!
```

**Right way** (SAFE):
```python
z_score = (x - x.expanding().mean()) / x.expanding().std()  # Only past data
```


In [ ]:
# Create normalized feature matrix using SAFE expanding-window z-scoring
# This is critical for walk-forward validation later

features_normalized = features_for_hmm.copy()

# For each feature, compute expanding mean and std (only using data up to time t)
for col in hmm_features:
    expanding_mean = features_for_hmm[col].expanding(min_periods=126).mean()
    expanding_std = features_for_hmm[col].expanding(min_periods=126).std()
    
    # Safe z-score: (x_t - mean up to t) / std up to t
    features_normalized[f"{col}_zscore"] = (features_for_hmm[col] - expanding_mean) / expanding_std

# Drop the original columns, keep only z-scored
zscore_cols = [f"{col}_zscore" for col in hmm_features]
features_normalized = features_normalized[zscore_cols]

# Drop NaNs from the expanding window warmup
features_normalized = features_normalized.dropna()

print(f"Z-scored feature matrix shape: {features_normalized.shape}")
print(f"\nZ-scored feature statistics (should be ~mean=0, std~1 asymptotically):")
print(features_normalized.describe().round(4))

print(f"\nFirst 5 rows (after expanding window warmup):")
print(features_normalized.head())

## Demonstrate the Lookahead Bias Difference

Show the difference between LEAKY and SAFE z-scoring.

In [ ]:
# Compare leaky vs safe z-scoring on one feature
example_feature = "vol_21d_avg"

# LEAKY: full-sample z-score (uses future data)
leaky_zscore = (features_for_hmm[example_feature] - features_for_hmm[example_feature].mean()) / features_for_hmm[example_feature].std()

# SAFE: expanding z-score (only uses past data)
expanding_mean = features_for_hmm[example_feature].expanding(min_periods=126).mean()
expanding_std = features_for_hmm[example_feature].expanding(min_periods=126).std()
safe_zscore = (features_for_hmm[example_feature] - expanding_mean) / expanding_std

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Plot comparison
axes[0].plot(features_for_hmm.index, leaky_zscore, label="LEAKY (full-sample stats)", alpha=0.7, color="red")
axes[0].plot(features_for_hmm.index, safe_zscore, label="SAFE (expanding stats)", alpha=0.7, color="green")
axes[0].set_title(f"Z-Score Comparison: {example_feature}")
axes[0].set_ylabel("Z-Score")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Difference plot
difference = (leaky_zscore - safe_zscore).dropna()
axes[1].plot(difference.index, difference, label="Leaky - Safe (bias)", alpha=0.7, color="purple")
axes[1].axhline(0, color="black", linestyle="-", linewidth=0.5, alpha=0.5)
axes[1].set_title("Lookahead Bias: Difference between Leaky and Safe Z-Score")
axes[1].set_ylabel("Bias (Z-score difference)")
axes[1].set_xlabel("Date")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Shade early period
early_cutoff = features_for_hmm.index[500]
for ax in axes:
    ax.axvspan(features_for_hmm.index[0], early_cutoff, alpha=0.05, color="red", label="Early period (more bias)")

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("✅ Lookahead Bias Demonstration:")
print(f"   • LEAKY z-score uses FUTURE data to normalize past observations")
print(f"   • SAFE z-score only uses data available at each point in time")
print(f"   • Bias is larger in early periods (fewer observations)")
print(f"   • This is why walk-forward validation REQUIRES re-fitting per fold")

## Save Phase 2 Output: Engineered Features

Save both raw and normalized features for Phase 3 (HMM training).

In [ ]:
# Save raw features
features_for_hmm.to_csv("data/features_raw.csv")
print("✅ Saved raw features to data/features_raw.csv")

# Save normalized features
features_normalized.to_csv("data/features_normalized.csv")
print("✅ Saved normalized features to data/features_normalized.csv")

# Save all engineered features (for reference)
features_df.to_csv("data/features_all.csv")
print("✅ Saved all engineered features to data/features_all.csv")

print(f"\n📊 Phase 2 Summary:")
print(f"   • Raw features shape: {features_for_hmm.shape}")
print(f"   • Normalized features shape: {features_normalized.shape}")
print(f"   • Date range: {features_normalized.index[0].date()} to {features_normalized.index[-1].date()}")
print(f"   • All features are z-scored using safe expanding windows (no lookahead bias)")

## Feature Engineering Summary

✅ **Momentum features** (direction of each asset at multiple horizons)
✅ **Volatility features** (stress level captured at multiple scales)
✅ **Cross-asset features** (correlation and average volatility)
✅ **VIX features** (market fear level and momentum)
✅ **Safe normalization** (expanding-window z-scoring, no lookahead bias)

**Key insight**: These 4 features distinguish regimes by capturing:
- **Bull**: Positive momentum, low volatility, low VIX, low correlation
- **Bear**: Negative momentum, moderate volatility, moderate VIX, increasing correlation
- **Crisis**: Mixed momentum, high volatility, high VIX, high correlation

---

## Next: Phase 3 - Regime Detection (HMM)

In the next phase, we'll:
1. Implement a **3-state Hidden Markov Model** using hmmlearn
2. Fit it to our engineered features
3. Label states as Bull/Bear/Crisis
4. Visualize regimes overlaid on price charts
5. Inspect the transition matrix

---